# 第 6 节：Monte Carlo 方法

---

## 📍 本节位置

```
Bellman/DP (05) → **Monte Carlo (06)** → TD Learning (07) → ...
                       ↑
                   你在这里
```

DP 需要完整的环境模型（$P$ 和 $R$）。**Monte Carlo 方法不需要模型**——它直接从经验（采样的完整 episode）中学习。

---

## 🎯 学习目标

1. 理解 MC 方法如何从经验中估计价值
2. 区分 First-Visit MC 和 Every-Visit MC
3. 推导增量均值更新公式
4. 理解 MC 的 bias-variance 特性
5. 实现 MC Prediction 和 MC Control
6. 理解 Exploring Starts 的必要性


## 1. 从 DP 到 MC：为什么需要从经验学习？

### DP 的局限

DP (Policy Iteration / Value Iteration) 需要：
- $P(s'|s,a)$：完整的状态转移概率
- $R(s,a)$：期望奖励

在大多数实际问题中，**我们没有这些信息**。

### MC 的核心思想

> **用采样回报的均值来估计期望回报。**

$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] \approx \frac{1}{N} \sum_{i=1}^{N} G_t^{(i)}$$

其中 $G_t^{(i)}$ 是第 $i$ 条 episode 中从状态 $s$ 出发的实际回报。

**关键**：我们不需要知道 $P$ 和 $R$，只需要完整的 episode！


## 2. MC 价值估计的两种变体

### First-Visit MC

对每个状态 $s$，**只使用它在每条 episode 中第一次出现**时的回报：

```
对于每条 episode:
    G ← 0
    对于 t = T-1, T-2, ..., 0:
        G ← R_{t+1} + γ·G
        如果 S_t 未在这条 episode 中更早出现过:
            将 G 加入 S_t 的回报列表
    V(s) ← mean(回报列表)
```

### Every-Visit MC

对每个状态 $s$，使用它在每条 episode 中**每次出现**时的回报：

```
对于每条 episode:
    G ← 0
    对于 t = T-1, T-2, ..., 0:
        G ← R_{t+1} + γ·G
        将 G 加入 S_t 的回报列表  # 每次都加入！
    V(s) ← mean(回报列表)
```

### 区别

| 方面 | First-Visit | Every-Visit |
|------|-------------|-------------|
| 数据量 | 少 | 多 |
| 偏差 | 无偏 | 有偏（episode 内状态相关） |
| 渐近收敛 | ✅ 收敛到 V^π | ✅ 收敛到 V^π |
| 实用性 | 更常用 | 在样本少时可能更好 |


## 3. 增量均值更新

### 为什么需要增量形式？

如果直接存储所有回报再取平均，需要 $O(N)$ 内存。

### 推导

设 $Q_n$ 是 $n$ 个值 $x_1, \ldots, x_n$ 的均值：

$$
\begin{aligned}
Q_{n+1} &= \frac{1}{n+1} \sum_{i=1}^{n+1} x_i \\
&= \frac{1}{n+1}\left(x_{n+1} + \sum_{i=1}^{n} x_i\right) \\
&= \frac{1}{n+1}\left(x_{n+1} + n \cdot Q_n\right) \\
&= Q_n + \frac{1}{n+1}(x_{n+1} - Q_n)
\end{aligned}
$$

### 一般形式

$$Q_{n+1} = Q_n + \alpha_n (x_{n+1} - Q_n)$$

- 当 $\alpha_n = 1/n$：普通均值
- 当 $\alpha_n = \alpha$（常数）：指数加权平均（用在非平稳问题）

这正是后续 **TD 学习** 中更新规则的前身！


In [ ]:
# 增量均值的 Python 验证
import numpy as np
np.random.seed(42)

# 生成随机数据
data = np.random.randn(100)
true_mean = np.mean(data)  # 真实均值

# 增量均值
Q = 0.0
incremental_means = []
for n, x in enumerate(data, 1):
    Q = Q + (1/n) * (x - Q)  # 普通均值
    # Q = Q + 0.1 * (x - Q)  # 常数步长（指数加权）
    incremental_means.append(Q)

print(f"真实均值: {true_mean:.4f}")
print(f"增量均值: {Q:.4f}")
print(f"最终误差: {abs(Q - true_mean):.6f}")

# 可视化收敛
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(incremental_means, linewidth=1, label='增量均值')
ax.axhline(true_mean, color='r', linestyle='--', label=f'真实均值 = {true_mean:.3f}')
ax.set_xlabel('样本数 n'); ax.set_ylabel('Q_n')
ax.set_title('增量均值收敛'); ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/06_incremental_mean.png'); plt.close()
print("✅ 图已保存")


## 4. MC Prediction 实现

In [ ]:
import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.envs.grid_world import GridWorld
from rl_course.utils.seeding import set_seed; set_seed(42)
from collections import defaultdict

def mc_prediction_first_visit(env, pi, gamma, n_episodes):
    '''First-Visit MC Prediction

    Args:
        env: 环境
        pi: 策略函数 pi(s) -> action (确定性) 或 pi[s] -> prob (数组)
        gamma: 折扣因子
        n_episodes: episode 数量

    Returns:
        V: 估计的状态价值 (nS,)
    '''
    nS = env.n_states
    returns_sum = np.zeros(nS)  # 回报累积和
    returns_count = np.zeros(nS)  # 回报计数
    V = np.zeros(nS)

    for ep in range(n_episodes):
        # 生成一条 episode
        episode = []
        state = env.reset()
        done = False
        while not done:
            if callable(pi):
                action = pi(state)
            else:
                action = np.random.choice(env.n_actions, p=pi[state]) if pi.ndim > 1 else pi[state]
            next_state, reward, done, _ = env.step(action)
            episode.append((state, action, reward))
            state = next_state

        # 从后向前计算回报
        G = 0.0
        visited_states = set()
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = reward + gamma * G

            # First-visit: 仅记录首次出现
            if state not in visited_states:
                visited_states.add(state)
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]

        if (ep + 1) % 100 == 0:
            print(f"  Episode {ep+1}/{n_episodes}")

    return V

# 在 GridWorld 上测试
gw = GridWorld(width=4, height=4, step_reward=-1.0, goal_reward=10.0, seed=42)
random_pi = np.ones((gw.n_states, gw.n_actions)) / gw.n_actions  # 均匀随机策略

V_mc = mc_prediction_first_visit(gw, random_pi, gamma=0.99, n_episodes=500)
print(f"\nMC 估计的 V:")
print(V_mc.reshape(4, 4))


## 5. 与 DP (真实值) 对比

MC 估计的价值函数与 DP 计算的真实值对比：

In [ ]:
# DP 计算真实 V^π
P_gw = gw.get_transition_matrix()
R_gw = gw.get_reward_matrix()
gamma = 0.99

# 解析解 V^π = (I - γP^π)^(-1) R^π
nS, nA = gw.n_states, gw.n_actions
P_pi = np.sum(random_pi[:, :, np.newaxis] * P_gw, axis=1)  # (nS, nS)
R_pi = np.sum(random_pi * R_gw, axis=1)  # (nS,)
V_true = np.linalg.solve(np.eye(nS) - gamma * P_pi, R_pi)

# 对比
print("MC 估计 vs 真实值:")
print(f"{'状态':<6} {'MC V':<10} {'True V':<10} {'误差':<10}")
for s in range(nS):
    print(f"{s:<6} {V_mc[s]:<10.4f} {V_true[s]:<10.4f} {abs(V_mc[s]-V_true[s]):<10.4f}")

error = np.max(np.abs(V_mc - V_true))
print(f"\n最大绝对误差: {error:.4f}")
print(f"均方根误差: {np.sqrt(np.mean((V_mc - V_true)**2)):.4f}")


## 6. MC 的 Bias-Variance 分析

### Bias（偏差）

MC 对 $V^\pi$ 的估计是**无偏**的（First-Visit）：

$$\mathbb{E}[\hat{V}_{MC}(s)] = V^\pi(s)$$

原因是 $G_t$ 本身就是 $V^\pi(s)$ 的无偏估计（$V^\pi(s) = \mathbb{E}[G_t | S_t=s]$）。

偏差来自 Every-Visit MC（同一条 episode 内的状态相关，导致有偏估计）。

### Variance（方差）

MC 的方差**大**！$G_t$ 是多个随机变量之和：

$$\text{Var}[G_t] = \text{Var}[R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots]$$

每一步的随机性（动作选择 + 状态转移 + 奖励）都会累积到 $G_t$ 中。

**方差来源**：
1. 环境随机性（$P$ 和 $R$）
2. 策略随机性（$\pi$）
3. 轨迹长度随机性

后续的 **TD 方法**通过引入 bootstrapping 来降低方差（但引入了偏差）。


In [ ]:
# 方差实验：多次运行 MC 估计
def mc_variance_experiment(env, pi, gamma, n_episodes, n_runs=30):
    '''多次运行 MC，计算 V 的方差'''
    all_V = []
    for run in range(n_runs):
        set_seed(run)
        V = mc_prediction_first_visit(env, pi, gamma, n_episodes)
        all_V.append(V)
    return np.array(all_V)

# 运行 30 次 MC（每次 200 episodes）
all_V_mc = mc_variance_experiment(gw, random_pi, gamma=0.99, n_episodes=200, n_runs=30)

# 计算每个状态的方差
variance_per_state = np.var(all_V_mc, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 方差热力图
im = axes[0].imshow(variance_per_state.reshape(4,4), cmap='Reds')
axes[0].set_title('V(s) 估计方差 (30 次运行)')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{variance_per_state.reshape(4,4)[i,j]:.1f}', ha='center', va='center')
plt.colorbar(im, ax=axes[0])

# MC vs True
axes[1].scatter(V_true, V_mc, alpha=0.5)
axes[1].plot([V_true.min(), V_true.max()], [V_true.min(), V_true.max()], 'r--')
axes[1].set_xlabel('True V'); axes[1].set_ylabel('MC V')
axes[1].set_title('MC vs True Value'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/06_mc_variance.png'); plt.close()
print("✅ MC 方差分析图已保存")


## 7. MC Control（MC 控制）

现在不仅估计价值，还要**改进策略**。

### MC with Exploring Starts (MC-ES)

核心挑战：如果总是贪心选择，某些状态-动作对永远不被尝试。

**Exploring Starts**：每条 episode 从一个随机的状态-动作对开始，保证所有对都被探索。


In [ ]:
from rl_course.agents.tabular import TabularMCAgent

# 使用我们封装好的 MC Agent
agent = TabularMCAgent(n_states=gw.n_states, n_actions=gw.n_actions, gamma=0.99, epsilon=0.1, seed=42)

# 手动训练循环（模拟 MC-ES）
n_episodes = 500
episode_returns = []

for ep in range(n_episodes):
    # Exploring start: 随机初始化状态
    state = np.random.randint(gw.n_states)
    gw.agent_pos = list(gw._state_to_pos[state])
    done = False

    episode_data = []
    while not done:
        action = agent.act(state, train=True)
        next_state, reward, done, _ = gw.step(action)
        episode_data.append((state, action, reward))
        state = next_state

    metrics = agent.update(episode_data)
    episode_returns.append(metrics['episode_return'])

# 可视化
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episode_returns, alpha=0.3, linewidth=0.5)
window = 20
if len(episode_returns) > window:
    smoothed = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(episode_returns)), smoothed, linewidth=2, color='red')
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('MC Control 训练曲线'); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/06_mc_control.png'); plt.close()
print("✅ MC Control 训练图已保存")

# 学习的 Q 表和策略
print("\n最优 Q 表:")
print(agent.Q[:4])  # 前 4 个状态
print(f"\n最优策略: {agent.policy.reshape(4,4)}")
print(f"(0=↑, 1=→, 2=↓, 3=←)")


## 8. 本节总结

### MC 的核心特征

| 特性 | MC | DP |
|------|-----|-----|
| 需要模型 | ❌ | ✅ |
| 数据来源 | 完整 episode 采样 | 模型 |
| Bias | 0（First-Visit）| 0 |
| Variance | 大 | 小 |
| 适用场景 | Episodic 任务 | 已知 MDP |

### 关键理解

1. **MC = 采样 + 平均**：用经验回报的均值估计期望回报
2. **需要完整 episode**：必须等 episode 结束才能计算回报
3. **高方差**：$G_t$ 累积了所有步骤的随机性
4. **无偏（First-Visit）**：每个 $G_t$ 是 $V^\pi(s)$ 的无偏估计

### 下一步

MC 的主要问题（高方差、必须等 episode 结束）由 **TD 方法**（下一节）解决。


## 9. 练习

1. 实现 Every-Visit MC Prediction，与 First-Visit 对比偏差
2. 证明：为什么 First-Visit MC 是无偏的？
3. 修改 MC-ES 为 on-policy ε-soft（无需 exploring starts）
4. 比较不同 ε 值对 MC Control 收敛速度的影响


---
*下一节：[07_td_learning.ipynb](07_td_learning.ipynb) — TD 学习（结合 MC 和 DP 的优点）*
